# Lección 10.1 — Cómo "ve" una computadora una imagen

> **Módulo 10 · Visión por computadora (CNN)** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 70–90 min (lección con ejercicios intercalados ≈ 55 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 2.1 (tipos de datos), Lección 9.2 (capas y profundidad)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar qué es un píxel y por qué una imagen es, para la computadora, una **tabla de números**.
- Distinguir imágenes en escala de grises (1 canal) y a color (3 canales RGB).
- Leer la **forma** (*shape*) de una imagen y de un lote de imágenes en NumPy y en PyTorch.
- Explicar por qué una red densa "píxel por píxel" es frágil y cara, y por qué necesitamos las convoluciones (Lección 10.2).

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.
>
> 🖥️ Todo corre en CPU en pocos minutos (también en Google Colab sin GPU). Si tienes GPU, se usará sola.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). La primera vez descarga MNIST (≈ 12 MB): 70,000 dígitos escritos a mano.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import datasets
from sklearn.datasets import load_sample_image
from sklearn.linear_model import LogisticRegression

plt.rcParams["figure.figsize"] = (8, 4.5)
mnist = datasets.MNIST(root="data", train=True, download=True)
mnist_test = datasets.MNIST(root="data", train=False, download=True)
print("✅ Todo listo")

---
## 1. 📖 La intuición: pintar por números

¿Recuerdas los cuadernos de "pintar por números"? Cada casilla trae un número y tú la rellenas con el color que le corresponde. Al terminar, aparece un dibujo.

Una computadora hace lo contrario: **no ve el dibujo, solo ve los números**.

```
Lo que tú ves:        Lo que ve la computadora:
    ███                 0   0 255 255 255   0
   █   █                0 255   0   0   0 255
   █   █                0 255   0   0   0 255
    ███                 0   0 255 255 255   0
```

Cada casilla es un **píxel** (*picture element*). En una imagen en blanco y negro, cada píxel es un número de **0 (negro) a 255 (blanco)**. Una foto es solo una tabla muy grande de esos números.

### 🔮 Ejercicio 1 — ¿Qué dibuja esta tabla?
Abajo hay una tabla de 7 × 7 números. **Antes de ejecutar**, mírala e intenta adivinar qué dibujo forma (los 255 son blancos).

✍️ *Mi apuesta:*

In [ ]:
dibujo = np.array([
    [  0,   0,   0,   0,   0,   0,   0],
    [  0, 255,   0,   0,   0, 255,   0],
    [  0,   0,   0,   0,   0,   0,   0],
    [  0,   0,   0, 255,   0,   0,   0],
    [255,   0,   0,   0,   0,   0, 255],
    [  0, 255, 255, 255, 255, 255,   0],
    [  0,   0,   0,   0,   0,   0,   0],
])
# ✏️ cambia algunos 0 por 255 (o por 128 = gris) y vuelve a ejecutar

plt.figure(figsize=(3, 3))
plt.imshow(dibujo, cmap="gray", vmin=0, vmax=255)
plt.title(f"Tabla de {dibujo.shape[0]} × {dibujo.shape[1]} números"); plt.axis("off"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Una **carita sonriente** 🙂: dos ojos (fila 1), una nariz (fila 3) y una sonrisa (filas 4–5). Para la computadora, sin embargo, solo son 49 números; "carita" es algo que **tú** reconoces. Enseñarle a reconocerlo es el objetivo de todo este módulo.
</details>

---
## 2. 📖 El concepto: una imagen es una tabla (matriz) de números

Veamos una imagen **real**: un dígito de MNIST, el conjunto de datos clásico de visión por computadora (70,000 dígitos escritos a mano, de 28 × 28 píxeles).

In [ ]:
imagen, etiqueta = mnist.data[0].numpy(), int(mnist.targets[0])

print("Forma (shape):", imagen.shape, "→ alto × ancho")
print("Tipo de dato  :", imagen.dtype, "→ enteros de 0 a 255")
print("Etiqueta      :", etiqueta)

fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
ax[0].imshow(imagen, cmap="gray"); ax[0].set_title("Lo que tú ves")
ax[1].imshow(imagen, cmap="gray")
for i in range(0, 28, 2):            # escribimos el valor de algunos píxeles encima
    for j in range(0, 28, 2):
        ax[1].text(j, i, imagen[i, j], ha="center", va="center", fontsize=5,
                   color="red" if imagen[i, j] > 128 else "dodgerblue")
ax[1].set_title("Lo que ve la computadora (1 de cada 4 píxeles)")
for a in ax: a.axis("off")
plt.show()

### ✋ Ejercicio 2 — Lee los píxeles
1. ¿Cuántos números hay en total en esta imagen?
2. ¿Qué valor tienen los píxeles del **fondo**? ¿Y los del **trazo** más intenso?
3. En MNIST el fondo es negro y el trazo blanco. Si "invirtieras" la imagen (fondo blanco, trazo negro), ¿qué cuenta harías con cada número?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. 28 × 28 = **784** números.
2. El fondo vale **0** (negro) y el trazo más intenso **255** (blanco). Los bordes del trazo tienen valores intermedios (grises), que suavizan el dibujo.
3. **255 − valor**. Lo comprobarás en la Sección 4.
</details>

### 📖 Imágenes a color: tres tablas apiladas
Una imagen a color tiene **tres** números por píxel: cuánto **rojo** (R), cuánto **verde** (G) y cuánto **azul** (B), cada uno de 0 a 255. Es como tener tres linternas de colores apuntando al mismo punto: mezclando sus intensidades obtienes cualquier color.

Cada una de esas tres tablas se llama **canal**. Veamos una foto incluida en scikit-learn (un templo en China).

In [ ]:
foto = load_sample_image("china.jpg")
print("Forma:", foto.shape, "→ alto × ancho × canales")
print("Píxel de la esquina superior izquierda (R, G, B):", foto[0, 0])

fig, ax = plt.subplots(1, 4, figsize=(16, 3.5))
ax[0].imshow(foto); ax[0].set_title("Foto original")
for k, (nombre, mapa) in enumerate([("Rojo (R)", "Reds"), ("Verde (G)", "Greens"), ("Azul (B)", "Blues")]):
    ax[k + 1].imshow(foto[:, :, k], cmap=mapa); ax[k + 1].set_title(f"Canal {nombre}")
for a in ax: a.axis("off")
plt.show()

### 🔮 Ejercicio 3 — Cuenta y mezcla
1. ¿Cuántos números necesita la computadora para guardar esta foto?
2. El píxel de la esquina vale `[174, 201, 231]`. ¿De qué color crees que es? (Pista: mira el cielo de la foto.)
3. ¿Qué color dará `(255, 255, 0)`? ¿Y `(0, 0, 0)`? ¿Y `(128, 128, 128)`?

✍️ *Mis predicciones:* Luego ejecuta la celda de abajo. ✏️ Prueba tus propias mezclas.

In [ ]:
print("Números en la foto:", f"{foto.size:,}")

mezclas = [(255, 0, 0), (255, 255, 0), (0, 0, 0), (128, 128, 128), (174, 201, 231)]   # ✏️ añade las tuyas
fig, ax = plt.subplots(1, len(mezclas), figsize=(2.2 * len(mezclas), 2))
for a, rgb in zip(ax, mezclas):
    a.imshow(np.array(rgb, dtype=np.uint8).reshape(1, 1, 3)); a.set_title(str(rgb), fontsize=9); a.axis("off")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. 427 × 640 × 3 = **819,840** números, ¡por una foto pequeña! Una foto de celular de 12 megapíxeles tiene unos 36 millones.
2. Un **azul cielo claro**: mucho azul, bastante verde y algo menos de rojo.
3. `(255, 255, 0)` = **amarillo** (rojo + verde con luz se suman, no es como mezclar pinturas); `(0, 0, 0)` = **negro** (todas las linternas apagadas); `(128, 128, 128)` = **gris** (los tres canales iguales siempre dan un gris).
</details>

---
## 3. 📖 Cómo funciona por dentro: forma, tensores y lotes

En la Lección 9.2 alimentabas una red con **filas de una tabla**. Con imágenes la "fila" tiene dimensiones extra. Para no perderte, piensa siempre en la **forma** (*shape*):

| Qué | Forma en NumPy / matplotlib | Forma en PyTorch |
|---|---|---|
| Una imagen en grises | `(alto, ancho)` | `(1, alto, ancho)` |
| Una imagen a color | `(alto, ancho, 3)` | `(3, alto, ancho)` |
| Un **lote** (*batch*) de N imágenes | `(N, alto, ancho, 3)` | `(N, 3, alto, ancho)` |

> 📝 PyTorch pone los **canales primero** (formato `C × H × W`, de *channels, height, width*). matplotlib y casi todas las librerías de imágenes los ponen al final (`H × W × C`). Confundirlos es uno de los errores más frecuentes.

Además, las redes prefieren números pequeños: casi siempre se **dividen los píxeles entre 255** para dejarlos entre 0 y 1 (es el escalado de la Lección 2.3).

In [ ]:
# De imagen (NumPy, H×W×C, 0–255) a tensor de PyTorch (C×H×W, 0–1)
tensor_foto = torch.tensor(foto).permute(2, 0, 1).float() / 255

print("NumPy  :", foto.shape, foto.dtype, "rango", foto.min(), "–", foto.max())
print("PyTorch:", tuple(tensor_foto.shape), tensor_foto.dtype, f"rango {tensor_foto.min():.0f} – {tensor_foto.max():.0f}")

### ✋ Ejercicio 4 — Adivina la forma
Escribe la forma en **PyTorch** de:
1. Un lote de **64** dígitos de MNIST.
2. Un lote de **32** fotos a color de **224 × 224** píxeles.
3. Un tensor de forma `(16, 3, 32, 32)`: ¿cuántas imágenes son, de qué tamaño y en color o en grises?

✍️ *Mis respuestas:*

In [ ]:
lote_mnist = torch.zeros(64, 1, 28, 28)       # ✏️ comprueba tus respuestas creando tensores
lote_fotos = torch.zeros(32, 3, 224, 224)
print(tuple(lote_mnist.shape), "→", lote_mnist.numel(), "números")
print(tuple(lote_fotos.shape), "→", f"{lote_fotos.numel():,}", "números")

<details><summary><b>👀 Ver respuesta</b></summary>

1. `(64, 1, 28, 28)`: 64 imágenes, 1 canal, 28 × 28. Son 50,176 números.
2. `(32, 3, 224, 224)`: ¡4,816,896 números en un solo lote! Por eso la visión por computadora necesita tanta memoria.
3. **16** imágenes **a color** (3 canales) de **32 × 32** píxeles (es el tamaño de CIFAR-10, que usarás en la Lección 10.5 y el proyecto).
</details>

---
## 4. 📖 Editar una imagen = hacer cuentas

Como una imagen es una tabla de números, **todo filtro de Instagram es aritmética**:

| Operación | Cuenta |
|---|---|
| Aclarar | sumar un número a todos los píxeles |
| Invertir (negativo) | `255 − píxel` |
| Recortar | quedarte con unas filas y columnas |
| Voltear (espejo) | leer las columnas al revés |
| Pasar a grises | promediar los tres canales |

In [ ]:
recorte = foto[50:300, 150:450]                     # filas 50–300, columnas 150–450
ediciones = {
    "Original (recorte)": recorte,
    "Negativo": 255 - recorte,
    "Espejo": recorte[:, ::-1],
    "Grises": recorte.mean(axis=2),
}
fig, ax = plt.subplots(1, 4, figsize=(16, 3.5))
for a, (titulo, img) in zip(ax, ediciones.items()):
    a.imshow(img.astype(np.uint8), cmap="gray"); a.set_title(titulo); a.axis("off")
plt.show()

### 🔮 Ejercicio 5 — La trampa del "aclarar"
Queremos aclarar la foto sumando **100** a cada píxel. Un píxel del cielo vale **200**. Los píxeles se guardan como `uint8` (enteros de 0 a 255, 1 byte cada uno).

¿Cuánto valdrá ese píxel después de sumar 100? ✍️ *Mi apuesta:*

In [ ]:
pixel = np.array([200], dtype=np.uint8)
print("200 + 100 con uint8   :", pixel + np.array([100], dtype=np.uint8))
print("200 + 100 con enteros :", pixel.astype(int) + 100)
print("Lo correcto (recortar):", np.clip(pixel.astype(int) + 100, 0, 255))

aclarada_mal = recorte + np.uint8(100)                                   # ❌
aclarada_bien = np.clip(recorte.astype(int) + 100, 0, 255).astype(np.uint8)   # ✅
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].imshow(aclarada_mal); ax[0].set_title("❌ Sumando en uint8")
ax[1].imshow(aclarada_bien); ax[1].set_title("✅ Con np.clip")
for a in ax: a.axis("off")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

¡**44**! Un `uint8` solo llega a 255; al pasarse, "da la vuelta" como el cuentakilómetros de un auto viejo: 300 − 256 = 44. Por eso el cielo, que debería quedar blanco, se vuelve **oscuro** y aparecen colores extraños en la imagen de la izquierda.

La solución: convertir a enteros más grandes (o a decimales), operar y **recortar** al rango 0–255 con `np.clip`. Es una razón más para trabajar con píxeles en decimales entre 0 y 1, como hace PyTorch.
</details>

---
## 5. 📖 ¿Qué pasa si le damos los píxeles "sueltos" a un modelo?

Lo más sencillo para clasificar imágenes es lo que ya sabes hacer: **aplanar** la tabla de 28 × 28 en una fila de 784 números y dársela a un modelo clásico o a una red densa (Lección 9.2).

```
28 × 28  ──aplanar──►  [p1, p2, p3, …, p784]  ──►  modelo  ──►  "es un 7"
```

Entrenemos una regresión logística (Lección 5.2) con 5,000 dígitos.

In [ ]:
X_tr = mnist.data[:5000].numpy().reshape(5000, -1) / 255      # aplanamos: (5000, 784)
y_tr = mnist.targets[:5000].numpy()
X_te = mnist_test.data[:2000].numpy()                          # guardamos el test como imágenes
y_te = mnist_test.targets[:2000].numpy()

modelo_pixeles = LogisticRegression(max_iter=300).fit(X_tr, y_tr)
print(f"Exactitud en prueba: {modelo_pixeles.score(X_te.reshape(2000, -1) / 255, y_te):.1%}")

¡Nada mal! Pero mira lo que pasa con una prueba muy injusta… o muy realista.

### 🔮 Ejercicio 6 — Mueve el dibujo
Vamos a **desplazar** cada dígito de prueba unos píxeles a la derecha. Para una persona, un 7 movido 3 píxeles sigue siendo un 7, sin duda.

¿Qué crees que pasará con la exactitud del modelo si desplazamos 1, 2, 3 y 4 píxeles? ✍️ *Mi apuesta:*

In [ ]:
for px in [0, 1, 2, 3, 4]:                   # ✏️ prueba otros desplazamientos
    movidas = np.roll(X_te, px, axis=2)        # axis=2 → mover hacia la derecha
    acc = modelo_pixeles.score(movidas.reshape(2000, -1) / 255, y_te)
    print(f"Desplazamiento {px} px → exactitud {acc:.1%}  " + "█" * int(acc * 40))

fig, ax = plt.subplots(1, 2, figsize=(5, 2.5))
ax[0].imshow(X_te[0], cmap="gray"); ax[0].set_title("Original")
ax[1].imshow(np.roll(X_te[0], 3, axis=1), cmap="gray"); ax[1].set_title("Movido 3 px")
for a in ax: a.axis("off")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

La exactitud se **desploma**: ≈87 % → ≈82 % (1 px) → ≈62 % (2 px) → ≈45 % (3 px) → ≈28 % (4 px).

¿Por qué? El modelo aprendió **"el píxel 400 suele estar encendido en los 7"**. Si mueves el dibujo, el trazo cae en **otros** píxeles y, para el modelo, es una imagen totalmente distinta. No entiende que "un trazo es un trazo, esté donde esté".
</details>

### 📖 Dos problemas del enfoque "píxel por píxel"
1. **No sabe que los vecinos importan ni que un patrón puede estar en cualquier lugar.** Para él, el píxel de la esquina y el del centro son columnas sin relación, como "edad" y "salario". Pierde toda la **estructura espacial**.
2. **Es carísimo.** Una red densa que recibe una foto de 224 × 224 × 3 y tiene una primera capa de 1,000 neuronas necesita un peso por cada par (píxel, neurona):

In [ ]:
entradas = 224 * 224 * 3
neuronas = 1_000
print(f"Pesos de la primera capa densa: {entradas * neuronas:,}")   # ✏️ prueba con 28*28 y 100 neuronas

### ✋ Ejercicio 7 — ¿Qué necesitaríamos?
Con lo que viste, describe con tus palabras **dos propiedades** que debería tener un buen "lector de imágenes".

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Mirar en vecindarios**: los píxeles cercanos forman bordes y trazos; hay que analizarlos juntos, no por separado.
2. **Reutilizar el mismo detector en toda la imagen**: un detector de "trazo vertical" debería servir igual arriba que abajo o a la izquierda que a la derecha. Así, si el dibujo se mueve, el detector lo sigue encontrando, y además se necesitan **muchísimos menos pesos** (un solo detector pequeño en lugar de un peso por píxel).

¡Eso es exactamente una **convolución**! Es el tema de la Lección 10.2. (150 millones de pesos se convierten en unos cientos.)
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| La imagen se ve con colores "rotos" o con manchas oscuras al aclararla | **Desbordamiento** de `uint8` (200 + 100 = 44) | Convertir a `float`, operar y usar `np.clip` |
| `imshow` da error o muestra algo rarísimo | Formato `C×H×W` (PyTorch) en lugar de `H×W×C` (matplotlib) | `tensor.permute(1, 2, 0)` antes de dibujar |
| PyTorch se queja: *expected 4D input* | Le pasaste **una** imagen `(C, H, W)` en vez de un lote `(N, C, H, W)` | `img.unsqueeze(0)` para crear un lote de 1 |
| La red entrena muy mal desde el inicio | Píxeles de 0 a 255 sin escalar | Dividir entre 255 (y normalizar, Lección 10.5) |
| Excelente en prueba, pésimo en el mundo real | Las fotos reales están movidas, giradas o con otra luz | Modelos convolucionales (10.2–10.3) y *data augmentation* (Lección 9.8) |

### 🔮 Ejercicio 8 — El error de la dimensión que falta
Tienes una sola imagen como tensor de forma `(3, 32, 32)` y la red espera un lote. ¿Qué forma tendrá después de `unsqueeze(0)`? ¿Y si luego quieres dibujarla con matplotlib?

In [ ]:
una = torch.rand(3, 32, 32)
print("Una imagen          :", tuple(una.shape))
print("Como lote de 1      :", tuple(una.unsqueeze(0).shape))
print("Lista para matplotlib:", tuple(una.permute(1, 2, 0).shape))

<details><summary><b>👀 Ver respuesta</b></summary>

`unsqueeze(0)` añade una dimensión al frente: `(1, 3, 32, 32)`, un **lote con una sola imagen**. Para dibujarla, los canales tienen que ir al final: `permute(1, 2, 0)` → `(32, 32, 3)`.
</details>

---
## 7. 📋 Ficha: la imagen como dato

| | Imagen digital |
|---|---|
| **Qué es** | Una tabla de píxeles; cada píxel tiene 1 número (grises) o 3 (RGB), de 0 a 255 |
| **Forma típica** | `H × W × C` en NumPy · `C × H × W` en PyTorch · lotes `N × C × H × W` |
| **Preparación mínima** | Pasar a decimales y escalar a 0–1 (y normalizar) |
| **Lo que la hace especial** | La **posición** importa: los vecinos forman bordes y formas, y el mismo patrón puede aparecer en cualquier lugar |
| **Por qué no basta un modelo "plano"** | Ignora la estructura espacial, se rompe si el objeto se mueve y necesita millones de pesos |
| **Lo que viene** | Convoluciones (10.2), *pooling* (10.3), arquitecturas famosas (10.4), *transfer learning* (10.5) |

## 8. 🧠 Resumen en 5 puntos
1. Para una computadora, una imagen es una **tabla de números**: cada píxel vale de 0 (negro) a 255 (blanco).
2. Las imágenes a color tienen **3 canales** (rojo, verde, azul) apilados.
3. En PyTorch, un lote de imágenes tiene forma **`(N, C, H, W)`**; los píxeles se pasan a decimales entre 0 y 1.
4. Editar imágenes es aritmética… y cuidado con el **desbordamiento** de `uint8`.
5. Un modelo que mira **píxeles sueltos** ignora a los vecinos, se rompe si el objeto se mueve y necesita demasiados pesos: por eso existen las **redes convolucionales**.

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** Una imagen en grises de 100 × 50 píxeles, ¿cuántos números tiene?
- a) 150 · b) 5,000 · c) 15,000 · d) 255

**2.** El píxel `(0, 0, 255)` es…
- a) Negro · b) Rojo · c) Azul · d) Blanco

**3.** Un tensor tiene forma `(8, 1, 28, 28)`. Es…
- a) Una imagen a color de 8 × 28 · b) 8 imágenes en grises de 28 × 28 · c) 28 imágenes a color · d) Una imagen de 8 canales

**4.** ¿Por qué un modelo "píxel por píxel" falla cuando el objeto se desplaza?
- a) Porque las imágenes desplazadas tienen más píxeles
- b) Porque aprendió qué píxeles **concretos** suelen estar encendidos, no la forma
- c) Porque la regresión logística no acepta imágenes
- d) Porque hay que dividir entre 255

**5.** ¿Qué resultado da `255 - imagen` en una imagen en grises?
- a) La deja igual · b) La aclara · c) Su negativo · d) La voltea como espejo

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** 100 × 50 = 5,000 (un solo canal).
2. **c)** Solo la "linterna" azul está encendida, al máximo.
3. **b)** N = 8 imágenes, C = 1 canal (grises), 28 × 28 píxeles.
4. **b)** Memorizó posiciones. Si la forma se mueve, los píxeles "importantes" cambian.
5. **c)** Lo blanco (255) pasa a 0 y lo negro (0) a 255.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Una persona reconoce a su perro en una foto aunque esté en una esquina, de lado o medio oculto tras un sillón. Menciona **tres** cosas que cambian entre dos fotos del mismo perro y que, para un modelo "píxel por píxel", harían que parezcan imágenes completamente distintas.

**B. Caso 💼** Una fábrica toma fotos de **1,920 × 1,080** píxeles a color de cada pieza para detectar defectos. Un analista propone aplanar las fotos y usar una red densa con una primera capa de 512 neuronas. Calcula (aproximadamente) cuántos pesos tendría esa capa y da **dos** argumentos para convencerlo de otra cosa.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Por ejemplo: la **posición** (esquina vs. centro), el **tamaño** (cerca o lejos), la **rotación o el punto de vista**, la **iluminación** (cambia todos los valores de los píxeles), el **fondo** y las **oclusiones** (partes tapadas). Todas cambian cuáles píxeles valen qué.

**B.** 1,920 × 1,080 × 3 ≈ 6.2 millones de entradas × 512 ≈ **3,200 millones de pesos** solo en la primera capa (unos 12 GB de memoria en decimales de 32 bits). Argumentos: (1) es imposible de entrenar bien con las fotos que tendrá una fábrica: con tantos pesos memorizaría (sobreajuste, Lección 4.4); (2) un defecto (una raya, una abolladura) puede aparecer **en cualquier parte** de la pieza; un detector local que recorra la imagen (convolución) lo encuentra donde esté, con muy pocos pesos. También podría reducir la resolución si los defectos son grandes.
</details>

### 9.3 🧩 Práctica integradora: "Mi primer filtro… y su límite"

Vas a construir, paso a paso, un pequeño "editor de fotos" y a medir cuánto le afecta a un modelo cada edición. Cada paso tiene su respuesta: **no avances sin revisarla**.

**Paso 1 🔮 — Grises "a ojo" vs. grises "humanos".** Nuestros ojos son más sensibles al verde que al azul. Por eso la conversión estándar a grises no es el promedio simple, sino `0.299·R + 0.587·G + 0.114·B`. ¿En qué zonas de la foto esperas ver más diferencia entre los dos métodos?

In [ ]:
promedio = foto.mean(axis=2)
humana = 0.299 * foto[:, :, 0] + 0.587 * foto[:, :, 1] + 0.114 * foto[:, :, 2]
diferencia = np.abs(promedio - humana)

fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
ax[0].imshow(promedio, cmap="gray"); ax[0].set_title("Promedio simple")
ax[1].imshow(humana, cmap="gray"); ax[1].set_title("Ponderado (humano)")
ax[2].imshow(diferencia, cmap="magma"); ax[2].set_title("Diferencia (claro = mayor)")
for a in ax: a.axis("off")
plt.show()
print(f"Diferencia máxima: {diferencia.max():.0f} niveles de gris · promedio: {diferencia.mean():.1f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Las mayores diferencias (hasta ≈28 niveles) aparecen en la **vegetación de la parte de abajo**, de color verde-amarillento y casi sin azul: el método "humano" le da mucho peso al verde y la ve **más clara** que el promedio simple. En el **cielo** pasa lo contrario, aunque más suave: el promedio cuenta el azul tanto como el verde y lo ve un poco más claro. En zonas grises o blancas los dos coinciden (si R = G = B, ambos dan lo mismo). Moraleja: incluso "pasar a grises" es una decisión de diseño.
</details>

**Paso 2 ✏️ — Umbral: de grises a blanco y negro.** Convierte el dígito a solo dos valores: píxeles encendidos (> `UMBRAL`) y apagados. Prueba `UMBRAL` = 10, 128 y 240. ¿Cuál conserva mejor la forma del dígito?

In [ ]:
UMBRAL = 128   # ✏️ prueba 10, 128, 240
digito = X_te[0]
binario = (digito > UMBRAL).astype(int)
print(f"Píxeles encendidos: {binario.sum()} de {binario.size}")
plt.figure(figsize=(3, 3)); plt.imshow(binario, cmap="gray"); plt.axis("off"); plt.title(f"Umbral = {UMBRAL}"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Con **10** el trazo queda grueso (entran también los bordes grises); con **240** se pierde parte del trazo y aparecen huecos; **128** es un buen equilibrio. Este "umbral" es la forma más simple de **segmentación** (separar objeto y fondo), que retomarás en la Lección 10.6.
</details>

**Paso 3 🔮 — Otras ediciones "inofensivas".** Para una persona, un dígito **un poco más oscuro** o **volteado de arriba abajo** sigue siendo… bueno, el oscuro sí. Antes de ejecutar, ordena estas ediciones de la que **menos** a la que **más** daño le hará al modelo `modelo_pixeles`:
- a) Oscurecer (multiplicar los píxeles por 0.6)
- b) Desplazar 2 px a la derecha
- c) Voltear de arriba abajo

✍️ *Mi orden:*

In [ ]:
ediciones_test = {
    "Ninguna": X_te,
    "a) Oscurecer ×0.6": X_te * 0.6,
    "b) Desplazar 2 px": np.roll(X_te, 2, axis=2),
    "c) Voltear arriba-abajo": X_te[:, ::-1, :],
}
for nombre, Xe in ediciones_test.items():
    acc = modelo_pixeles.score(Xe.reshape(2000, -1) / 255, y_te)
    print(f"{nombre:<24} → exactitud {acc:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Orden: **a) → b) → c)**. Oscurecer afecta poco (≈87 % → ≈82 %): todos los píxeles bajan en la misma proporción y el patrón de "qué píxeles se encienden" se conserva. Desplazar 2 px ya hace mucho daño (≈62 %). Voltear es lo peor (≈29 %), y con razón: un 7 volteado ya no es un 7 para nadie (y un 6 volteado empieza a parecerse a un 9 en espejo). **Ojo:** no toda edición "conserva la etiqueta"; esto importará mucho al hacer *data augmentation* (Lección 10.5).
</details>

**Paso 4 ✋ — Conclusión.** Con tus resultados de los pasos 1–3 y de la Sección 5, escribe en dos frases qué le pedirías a un "modelo de visión" ideal.

✍️ *Mi conclusión:*

<details><summary><b>👀 Ver respuesta</b></summary>

Algo como: *"Un buen modelo de visión debería reconocer las **formas** (trazos, bordes, partes) estén donde estén y con distinta iluminación, en lugar de memorizar qué píxel concreto vale cuánto. Además, debería poder hacerlo con pocos parámetros para no memorizar."* En la próxima lección verás la herramienta que lo hace posible: la **convolución**.
</details>

---
## 📚 Para profundizar
- 🎮 **Interactivo:** *Image Kernels Explained Visually* (setosa.io): ya te prepara para la próxima lección.
- 📺 **Video:** 3Blue1Brown, *But what is a neural network?* (capítulo 1): usa justo MNIST.
- 📘 **Documentación:** PyTorch, `torchvision.transforms` (cómo convertir imágenes en tensores).

---
⬅️ **Anterior:** [9.10 Proyecto del módulo: El clasificador del armario 👕](../m09-redes-neuronales/9.10-proyecto-del-modulo.md) · ➡️ **Siguiente:** [10.2 Convoluciones: filtros que detectan patrones](10.2-convoluciones.ipynb)